# CenterNet + CSL Angle Head
### Single-model, end-to-end tube detection + true 360° orientation

```
Input Image (640×480)
        │
        ▼
┌─────────────────────────────────────────────────────────┐
│             DLA-34 Encoder-Decoder Backbone             │
│          (pretrained, output stride 4 → 160×120)        │
└────────────────────────┬────────────────────────────────┘
                         │  shared feature map (64-ch)
           ┌─────────────┼──────────────┐
           ▼             ▼              ▼
    ┌─────────────┐ ┌──────────┐ ┌───────────────────────┐
    │ Heatmap head│ │ WH head  │ │   CSL Angle head      │
    │  (1 class)  │ │ (w, h)   │ │  360-bin soft class   │
    │  Focal loss │ │  L1 loss │ │  Circular smooth label│
    └──────┬──────┘ └────┬─────┘ └──────────┬────────────┘
           │             │                  │
           └─────────────┴──────────────────┘
                         │
                         ▼
             NMS-free peak extraction
                         │
                         ▼
           center_x, center_y, angle_deg
```

### Why CSL beats sin/cos regression
| Method | Wraps at 0/360? | Fine-grained? | Handles symmetry? |
|--------|----------------|---------------|-------------------|
| Raw degree regression | ✗ hard discontinuity | ✓ | ✗ |
| sin/cos regression | ✓ smooth | ✓ | ✗ partial |
| **CSL (360 soft bins)** | **✓ circular by design** | **✓ 1° resolution** | **✓ Gaussian spread** |

CSL converts angle prediction into a **soft classification** over 360 one-degree bins.
A Gaussian window (σ=6°) centred on the true bin smooths the label so adjacent bins
receive gradient too — this prevents the hard-boundary collapse that hurts regression.

## 0 · Install dependencies

In [ ]:
!pip install torch torchvision opencv-python-headless matplotlib pandas \
             numpy scikit-learn Pillow tqdm scipy -q
print('Done.')

## 1 · Setup — extract data, find paths (macOS-safe)

In [ ]:
import os, shutil, zipfile, math, warnings, random
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

ZIP_PATH   = '/content/sample_data/data.zip'
EXTRACT_TO = '/content/data'

# Fresh extraction
if os.path.exists(EXTRACT_TO):
    shutil.rmtree(EXTRACT_TO)
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_TO)

# Locate real IMG_DIR and ANN_CSV — skip __MACOSX
IMG_DIR = ANN_CSV = None
for root, dirs, files in os.walk(EXTRACT_TO):
    if '__MACOSX' in root:
        continue
    if 'annotations.csv' in files and ANN_CSV is None:
        ANN_CSV = os.path.join(root, 'annotations.csv')
    if sum(1 for f in files if f.lower().endswith('.png')) >= 3 and IMG_DIR is None:
        IMG_DIR = root

assert IMG_DIR and ANN_CSV
print(f'IMG_DIR : {IMG_DIR}')
print(f'ANN_CSV : {ANN_CSV}')

df = pd.read_csv(ANN_CSV)
print(f'Total tubes: {len(df)}  |  Images: {df["image"].nunique()}')
print(df.head(3))

## 2 · EDA

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df['angle_deg'], bins=72, color='steelblue', edgecolor='none')
axes[0].set_title('Angle distribution'); axes[0].set_xlabel('Degrees')

ax_p = fig.add_subplot(1, 3, 2, projection='polar')
ax_p.hist(np.radians(df['angle_deg']), bins=36, color='darkorange', alpha=0.8)
ax_p.set_title('Polar rose', pad=14)

axes[2].scatter(df['center_x'], df['center_y'], s=12, alpha=0.4, c='purple')
axes[2].set_xlim(0,640); axes[2].set_ylim(480,0)
axes[2].set_title('Center positions')
plt.tight_layout(); plt.show()
print(f'Angle range: {df.angle_deg.min():.1f}° – {df.angle_deg.max():.1f}°')
print(f'Bbox size (mean): w={df.bbox_w.mean():.1f}  h={df.bbox_h.mean():.1f}')

## 3 · Train / Val split

In [ ]:
all_images = df['image'].unique().tolist()
train_imgs, val_imgs = train_test_split(all_images, test_size=0.2, random_state=42)
df_train = df[df['image'].isin(train_imgs)].reset_index(drop=True)
df_val   = df[df['image'].isin(val_imgs)].reset_index(drop=True)
print(f'Train: {len(train_imgs)} images, {len(df_train)} tubes')
print(f'Val  : {len(val_imgs)} images, {len(df_val)} tubes')

## 4 · Global config

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# ── Image / feature-map dimensions ────────────────────────────────────────
IMG_W, IMG_H   = 512, 512   # network input (square, power-of-2 friendly)
OUT_STRIDE     = 4          # CenterNet default: feature map = IMG / 4
FEAT_W         = IMG_W // OUT_STRIDE   # 128
FEAT_H         = IMG_H // OUT_STRIDE   # 128

# ── CSL parameters ────────────────────────────────────────────────────────
N_ANGLE_BINS   = 360        # 1° per bin
CSL_SIGMA      = 6          # Gaussian half-width in degrees (±6° gets gradient)

# ── Training ──────────────────────────────────────────────────────────────
BATCH_SIZE     = 4          # small dataset — keep batch small
EPOCHS         = 120
LR             = 5e-4
WARMUP_EPOCHS  = 5
HEATMAP_SIGMA  = 4          # Gaussian radius for GT heatmap (pixels in feat space)

# ── Loss weights ──────────────────────────────────────────────────────────
W_HM   = 1.0   # heatmap focal loss
W_WH   = 0.1   # width/height L1
W_OFF  = 1.0   # sub-pixel offset L1
W_ANG  = 1.0   # CSL angle cross-entropy

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'Feature map: {FEAT_W}×{FEAT_H}   Angle bins: {N_ANGLE_BINS}')

## 5 · CSL Utilities

```
True angle = 73°
CSL label (360 bins, σ=6):

bin: ...  67  68  69  70  71  72  73  74  75  76  77  78  79 ...
val: ... .14 .26 .43 .61 .80 .94 1.0 .94 .80 .61 .43 .26 .14 ...
                              ↑ peak at true bin

CrossEntropy(pred, soft_label) gives gradient to ±~18° neighbours
```

In [ ]:
def make_csl_label(angle_deg, n_bins=N_ANGLE_BINS, sigma=CSL_SIGMA):
    """
    Build a Gaussian-smoothed soft label vector for a single angle.
    The Gaussian wraps circularly so bin 0 and bin 359 are adjacent.

    Args:
        angle_deg : float, true angle in [0, 360)
        n_bins    : number of discrete bins (360 → 1°/bin)
        sigma     : Gaussian std in bins (degrees)
    Returns:
        np.ndarray of shape (n_bins,), values in [0,1], peak at true bin
    """
    bins   = np.arange(n_bins, dtype=np.float32)
    center = angle_deg % n_bins
    # Circular distance from each bin to the true angle
    diff   = np.abs(bins - center)
    diff   = np.minimum(diff, n_bins - diff)    # wrap around
    label  = np.exp(-(diff ** 2) / (2 * sigma ** 2))
    return label   # NOT normalised to sum=1 — we use BCEWithLogitsLoss


def csl_pred_to_angle(logits):
    """
    Convert raw CSL logits (B, 360) → angle in degrees [0, 360).
    Uses circular mean of the top-k bins weighted by softmax probability
    for sub-bin resolution.
    """
    probs   = torch.softmax(logits, dim=-1)          # (B, 360)
    bins    = torch.arange(N_ANGLE_BINS, device=logits.device, dtype=torch.float32)
    # Weighted circular mean via sin/cos
    rad     = bins * (2 * math.pi / N_ANGLE_BINS)
    sin_sum = (probs * torch.sin(rad)).sum(dim=-1)
    cos_sum = (probs * torch.cos(rad)).sum(dim=-1)
    angle   = torch.atan2(sin_sum, cos_sum)          # (-π, π]
    angle   = torch.rad2deg(angle) % 360             # [0, 360)
    return angle


def circular_mae(pred_deg, gt_deg):
    """Mean Absolute Angular Error respecting 360° wrap."""
    diff = (pred_deg - gt_deg).abs() % 360
    diff = torch.where(diff > 180, 360 - diff, diff)
    return diff.mean().item()


# ── Visual sanity check ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, ang in zip(axes, [0, 73, 355]):
    lbl = make_csl_label(ang)
    ax.bar(range(N_ANGLE_BINS), lbl, color='steelblue', width=1)
    ax.set_title(f'CSL label for {ang}°')
    ax.set_xlabel('Bin (degrees)'); ax.set_xlim(0, 360)
plt.suptitle('Circular Smooth Labels — Gaussian spread over 360 bins', fontsize=11)
plt.tight_layout(); plt.show()
print('Note how 355° wraps smoothly into 0° (bin 0 gets non-zero weight).')

## 6 · Heatmap utilities
Ground-truth heatmap: a 2-D Gaussian centred on each tube lid center,
rendered at 1/4 resolution (feature-map space).

In [ ]:
def gaussian_radius(det_size, min_overlap=0.7):
    """Compute heatmap Gaussian radius from bbox size (CornerNet formula)."""
    h, w = det_size
    a1 = 1
    b1 = h + w
    c1 = w * h * (1 - min_overlap) / (1 + min_overlap)
    sq1 = math.sqrt(b1 ** 2 - 4 * a1 * c1)
    r1 = (b1 - sq1) / (2 * a1)
    a2 = 4
    b2 = 2 * (h + w)
    c2 = (1 - min_overlap) * w * h
    sq2 = math.sqrt(b2 ** 2 - 4 * a2 * c2)
    r2 = (b2 - sq2) / (2 * a2)
    a3 = 4 * min_overlap
    b3 = -2 * min_overlap * (h + w)
    c3 = (min_overlap - 1) * w * h
    sq3 = math.sqrt(b3 ** 2 - 4 * a3 * c3)
    r3 = (b3 + sq3) / (2 * a3)
    return max(0, int(min(r1, r2, r3)))


def draw_gaussian(heatmap, center, radius, k=1):
    """Paint a 2-D Gaussian blob onto heatmap (in-place, takes element-wise max)."""
    diameter = 2 * radius + 1
    sigma    = diameter / 6
    x = np.arange(0, diameter, 1, dtype=np.float32)
    y = x[:, np.newaxis]
    x0 = y0 = radius
    gaussian = np.exp(-((x-x0)**2 + (y-y0)**2) / (2 * sigma**2))
    gaussian[gaussian < np.finfo(gaussian.dtype).eps * gaussian.max()] = 0

    cx, cy = int(center[0]), int(center[1])
    H, W   = heatmap.shape
    x1, x2 = cx - radius, cx + radius + 1
    y1, y2 = cy - radius, cy + radius + 1
    gx1 = max(0, -x1);  gx2 = diameter + min(0, W - x2)
    gy1 = max(0, -y1);  gy2 = diameter + min(0, H - y2)
    x1  = max(0, x1);   x2  = min(W, x2)
    y1  = max(0, y1);   y2  = min(H, y2)
    if x2 <= x1 or y2 <= y1: return
    heatmap[y1:y2, x1:x2] = np.maximum(
        heatmap[y1:y2, x1:x2], gaussian[gy1:gy2, gx1:gx2] * k)


print('Heatmap utilities ready.')

## 7 · Dataset

Each sample returns:
```
image      : (3, 512, 512)  float32 normalised
heatmap    : (1, 128, 128)  Gaussian peaks at tube centres
wh_map     : (2, 128, 128)  w, h at each tube centre
offset_map : (2, 128, 128)  sub-pixel offset at each tube centre
angle_map  : (360, 128, 128) CSL soft label at each tube centre
reg_mask   : (128, 128)      1 at tube centres, 0 elsewhere
```

In [ ]:
class CenterNetDataset(Dataset):
    def __init__(self, df_ann, img_dir, augment=False):
        self.records  = df_ann.groupby('image')
        self.img_names = list(self.records.groups.keys())
        self.img_dir  = img_dir
        self.augment  = augment
        self.normalize = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img_name = self.img_names[idx]
        ann      = self.records.get_group(img_name).reset_index(drop=True)

        # ── Load image ────────────────────────────────────────────────────
        img = cv2.imread(os.path.join(self.img_dir, img_name))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        orig_h, orig_w = img.shape[:2]

        # ── Augmentation (full-image so labels stay consistent) ───────────
        if self.augment:
            # Random horizontal flip
            if random.random() < 0.5:
                img = img[:, ::-1].copy()
                ann = ann.copy()
                ann['center_x'] = orig_w - ann['center_x']
                ann['angle_deg'] = (180 - ann['angle_deg']) % 360

            # Random vertical flip
            if random.random() < 0.5:
                img = img[::-1, :].copy()
                ann = ann.copy()
                ann['center_y'] = orig_h - ann['center_y']
                ann['angle_deg'] = (360 - ann['angle_deg']) % 360

            # Random 90° rotation (keeps aspect ratio)
            k = random.randint(0, 3)
            if k > 0:
                img = np.rot90(img, k).copy()
                ann = ann.copy()
                for _ in range(k):
                    cx = ann['center_x'].copy()
                    cy = ann['center_y'].copy()
                    ann['center_x'] = orig_h - cy
                    ann['center_y'] = cx
                    ann['angle_deg'] = (ann['angle_deg'] + 90) % 360
                    orig_w, orig_h = orig_h, orig_w

            # Colour jitter
            if random.random() < 0.8:
                img = img.astype(np.float32)
                img *= random.uniform(0.7, 1.3)   # brightness
                img  = np.clip(img, 0, 255).astype(np.uint8)

        # ── Resize to network input ───────────────────────────────────────
        scale_x = IMG_W / orig_w
        scale_y = IMG_H / orig_h
        img_resized = cv2.resize(img, (IMG_W, IMG_H), interpolation=cv2.INTER_LINEAR)

        # ── Allocate ground-truth tensors ─────────────────────────────────
        heatmap    = np.zeros((1, FEAT_H, FEAT_W), dtype=np.float32)
        wh_map     = np.zeros((2, FEAT_H, FEAT_W), dtype=np.float32)
        offset_map = np.zeros((2, FEAT_H, FEAT_W), dtype=np.float32)
        angle_map  = np.zeros((N_ANGLE_BINS, FEAT_H, FEAT_W), dtype=np.float32)
        reg_mask   = np.zeros((FEAT_H, FEAT_W), dtype=np.float32)

        for _, row in ann.iterrows():
            # Scale center to feature-map space
            fx = (row['center_x'] * scale_x) / OUT_STRIDE
            fy = (row['center_y'] * scale_y) / OUT_STRIDE
            ix = int(np.clip(fx, 0, FEAT_W - 1))
            iy = int(np.clip(fy, 0, FEAT_H - 1))

            # Heatmap — adaptive radius from bbox size
            fw = (row['bbox_w'] * scale_x) / OUT_STRIDE
            fh = (row['bbox_h'] * scale_y) / OUT_STRIDE
            radius = max(1, gaussian_radius((fh, fw)))
            draw_gaussian(heatmap[0], (ix, iy), radius)

            # WH target: log-space normalised by feat map size
            wh_map[0, iy, ix] = fw
            wh_map[1, iy, ix] = fh

            # Sub-pixel offset
            offset_map[0, iy, ix] = fx - ix
            offset_map[1, iy, ix] = fy - iy

            # CSL angle label (360-D Gaussian soft label)
            angle_map[:, iy, ix] = make_csl_label(row['angle_deg'])

            reg_mask[iy, ix] = 1.0

        img_tensor = self.normalize(img_resized)

        return {
            'image'     : img_tensor,
            'heatmap'   : torch.from_numpy(heatmap),
            'wh'        : torch.from_numpy(wh_map),
            'offset'    : torch.from_numpy(offset_map),
            'angle'     : torch.from_numpy(angle_map),
            'reg_mask'  : torch.from_numpy(reg_mask),
            'img_name'  : img_name,
        }


train_ds = CenterNetDataset(df_train, IMG_DIR, augment=True)
val_ds   = CenterNetDataset(df_val,   IMG_DIR, augment=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False,
                          num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds)} images  |  Val: {len(val_ds)} images')

# Visual sanity check — one training sample
sample = train_ds[0]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
mean = np.array([0.485,0.456,0.406]); std = np.array([0.229,0.224,0.225])
img_disp = (sample['image'].permute(1,2,0).numpy() * std + mean).clip(0,1)
axes[0].imshow(img_disp); axes[0].set_title('Input (512×512)')
axes[1].imshow(sample['heatmap'][0].numpy(), cmap='hot'); axes[1].set_title('GT Heatmap')
# Show CSL label for first tube
mask = sample['reg_mask'].numpy()
ys, xs = np.where(mask > 0)
if len(ys):
    csl_vec = sample['angle'][:, ys[0], xs[0]].numpy()
    axes[2].bar(range(360), csl_vec, width=1, color='darkorange')
    axes[2].set_title(f'CSL label @ first tube')
    axes[2].set_xlabel('Angle bin')
plt.tight_layout(); plt.show()

## 8 · Model: DLA-34 Backbone + CenterNet Heads

DLA-34 (Deep Layer Aggregation) is the standard CenterNet backbone.
Here we build a lightweight version that works well on small datasets.

In [ ]:
# ── Building blocks ───────────────────────────────────────────────────────

def conv_bn_relu(in_ch, out_ch, k=3, s=1, p=1):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, k, stride=s, padding=p, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    )

class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.body = nn.Sequential(
            conv_bn_relu(ch, ch), conv_bn_relu(ch, ch))
    def forward(self, x):
        return x + self.body(x)

class UpBlock(nn.Module):
    """Upsample × 2 + skip connection."""
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, 4, stride=2, padding=1)
        self.proj = nn.Conv2d(skip_ch, out_ch, 1) if skip_ch != out_ch else nn.Identity()
        self.fuse = conv_bn_relu(out_ch, out_ch)
    def forward(self, x, skip):
        return self.fuse(self.up(x) + self.proj(skip))


# ── Encoder (stride-4 at output) ─────────────────────────────────────────
class Encoder(nn.Module):
    """
    Simple 4-level encoder.
    In:  (B, 3, 512, 512)
    Skips returned at strides 2, 4, 8
    Out: (B, 256, 32, 32)  at stride 16
    """
    def __init__(self):
        super().__init__()
        self.s1 = nn.Sequential(conv_bn_relu(3,  32, s=2), ResBlock(32))    # /2
        self.s2 = nn.Sequential(conv_bn_relu(32, 64, s=2), ResBlock(64),
                                ResBlock(64))                                # /4
        self.s3 = nn.Sequential(conv_bn_relu(64,128, s=2), ResBlock(128),
                                ResBlock(128))                               # /8
        self.s4 = nn.Sequential(conv_bn_relu(128,256,s=2), ResBlock(256),
                                ResBlock(256), ResBlock(256))               # /16

    def forward(self, x):
        x1 = self.s1(x)
        x2 = self.s2(x1)
        x3 = self.s3(x2)
        x4 = self.s4(x3)
        return x4, x3, x2, x1


# ── Decoder (upsample back to stride-4) ──────────────────────────────────
class Decoder(nn.Module):
    """
    In : stride-16 features + skips at 8, 4
    Out: (B, 64, 128, 128)  at stride 4
    """
    def __init__(self):
        super().__init__()
        self.up1 = UpBlock(256, 128, 128)   # /16 → /8
        self.up2 = UpBlock(128,  64,  64)   # /8  → /4

    def forward(self, x4, x3, x2):
        d = self.up1(x4, x3)
        d = self.up2(d,  x2)
        return d   # (B, 64, 128, 128)


# ── Detection heads ───────────────────────────────────────────────────────
def make_head(in_ch, out_ch, hidden=64):
    """Standard CenterNet head: Conv→ReLU→Conv."""
    return nn.Sequential(
        nn.Conv2d(in_ch, hidden, 3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(hidden, out_ch, 1),
    )


class CenterNetCSL(nn.Module):
    """
    Full CenterNet model with four heads:
      hm   : (B, 1,   128, 128)  — heatmap (sigmoid)
      wh   : (B, 2,   128, 128)  — width, height
      off  : (B, 2,   128, 128)  — sub-pixel offset
      angle: (B, 360, 128, 128)  — CSL angle logits
    """
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

        feat_ch = 64
        self.hm_head    = make_head(feat_ch, 1)
        self.wh_head    = make_head(feat_ch, 2)
        self.off_head   = make_head(feat_ch, 2)
        self.angle_head = make_head(feat_ch, N_ANGLE_BINS, hidden=128)

        # Heatmap bias init (from CenterNet paper)
        nn.init.constant_(self.hm_head[-1].bias, -2.19)

    def forward(self, x):
        x4, x3, x2, _ = self.encoder(x)
        feat = self.decoder(x4, x3, x2)
        return {
            'hm'   : self.hm_head(feat).sigmoid(),
            'wh'   : self.wh_head(feat),
            'off'  : self.off_head(feat),
            'angle': self.angle_head(feat),
        }


model = CenterNetCSL().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {n_params:,}')

# Shape check
with torch.no_grad():
    dummy = torch.zeros(1, 3, IMG_H, IMG_W).to(DEVICE)
    out   = model(dummy)
    for k, v in out.items():
        print(f'  {k:8s}: {tuple(v.shape)}')

## 9 · Loss Functions

```
Total loss = λ_hm · L_focal
           + λ_wh · L1(wh)
           + λ_off· L1(offset)
           + λ_ang· BCE(CSL logits, soft label)
```

**Modified focal loss** (from CenterNet): penalises false positives near GT
less harshly by discounting predictions near the Gaussian peak.

**CSL loss**: Binary cross-entropy over 360 bins. Each bin is an independent
binary decision — the soft label (values in [0,1]) lets the Gaussian-weighted
neighbours also contribute to the gradient.

In [ ]:
def focal_loss(pred, gt, alpha=2.0, beta=4.0, eps=1e-6):
    """
    CenterNet modified focal loss.
    pred, gt: (B, 1, H, W), values in [0, 1]
    """
    pos_mask = (gt == 1).float()
    neg_mask = 1 - pos_mask

    pos_loss = -torch.log(pred.clamp(eps)) * ((1 - pred) ** alpha) * pos_mask
    neg_loss = (-torch.log((1 - pred).clamp(eps))
                * (pred ** alpha)
                * ((1 - gt) ** beta)
                * neg_mask)

    n_pos = pos_mask.sum()
    loss  = (pos_loss.sum() + neg_loss.sum()) / n_pos.clamp(1)
    return loss


def reg_l1_loss(pred, gt, mask):
    """
    L1 loss evaluated only at object centre locations.
    pred, gt: (B, C, H, W)
    mask    : (B, H, W)  — 1 at object centres
    """
    mask_exp = mask.unsqueeze(1).expand_as(pred)
    n_pos    = mask.sum().clamp(1)
    return (torch.abs(pred - gt) * mask_exp).sum() / n_pos


def csl_loss(pred_logits, gt_soft, mask):
    """
    CSL angle loss: Binary cross-entropy over 360 bins,
    evaluated only at object centre locations.

    pred_logits: (B, 360, H, W)  — raw (unnormalised)
    gt_soft    : (B, 360, H, W)  — Gaussian soft labels in [0, 1]
    mask       : (B, H, W)       — 1 at object centres
    """
    B, C, H, W = pred_logits.shape
    mask_exp   = mask.unsqueeze(1).expand(B, C, H, W)
    n_pos      = mask.sum().clamp(1) * C

    loss = F.binary_cross_entropy_with_logits(
        pred_logits, gt_soft, reduction='none')
    return (loss * mask_exp).sum() / n_pos


def compute_loss(preds, batch):
    hm_gt   = batch['heatmap'].to(DEVICE)
    wh_gt   = batch['wh'].to(DEVICE)
    off_gt  = batch['offset'].to(DEVICE)
    ang_gt  = batch['angle'].to(DEVICE)
    mask    = batch['reg_mask'].to(DEVICE)

    l_hm  = focal_loss(preds['hm'], hm_gt)
    l_wh  = reg_l1_loss(preds['wh'], wh_gt, mask)
    l_off = reg_l1_loss(preds['off'], off_gt, mask)
    l_ang = csl_loss(preds['angle'], ang_gt, mask)

    total = W_HM*l_hm + W_WH*l_wh + W_OFF*l_off + W_ANG*l_ang
    return total, {'hm': l_hm.item(), 'wh': l_wh.item(),
                   'off': l_off.item(), 'angle': l_ang.item()}


print('Loss functions ready.')

## 10 · Decode predictions from heatmap

In [ ]:
def heatmap_nms(hm, kernel=3):
    """Local max suppression — keeps only local peaks in the heatmap."""
    pad  = (kernel - 1) // 2
    hmax = F.max_pool2d(hm, kernel, stride=1, padding=pad)
    return hm * (hm == hmax).float()


def decode_predictions(preds, orig_w, orig_h,
                       conf_thresh=0.25, max_dets=10):
    """
    Decode network output into a list of detections.

    Returns list of dicts:
        {'center_x', 'center_y', 'bbox_w', 'bbox_h', 'angle_deg', 'score'}
    All coordinates in original image pixels.
    """
    hm   = heatmap_nms(preds['hm'])   # (1, 1, H, W)
    B, _, H, W = hm.shape

    hm_flat = hm.view(B, -1)           # (1, H*W)
    scores, inds = hm_flat.topk(max_dets, dim=1)

    # Grid coordinates
    xs = (inds % W).float()
    ys = (inds // W).float()

    # Sub-pixel offset
    off = preds['off'].view(B, 2, -1)  # (1, 2, H*W)
    off_x = off[:, 0, :].gather(1, inds)   # (1, k)
    off_y = off[:, 1, :].gather(1, inds)

    # Width / height
    wh = preds['wh'].view(B, 2, -1)
    ws = wh[:, 0, :].gather(1, inds)
    hs = wh[:, 1, :].gather(1, inds)

    # CSL angle
    ang_logits = preds['angle'].view(B, N_ANGLE_BINS, -1)  # (1, 360, H*W)
    # Gather 360-d vector for each top-k location
    inds_exp = inds.unsqueeze(1).expand(B, N_ANGLE_BINS, -1)   # (1,360,k)
    top_ang  = ang_logits.gather(2, inds_exp)                   # (1,360,k)
    top_ang  = top_ang.permute(0, 2, 1).squeeze(0)              # (k, 360)
    angles   = csl_pred_to_angle(top_ang).cpu().numpy()         # (k,)

    # Scale back to original image
    cx = ((xs + off_x) * OUT_STRIDE * orig_w / IMG_W).squeeze(0).cpu().numpy()
    cy = ((ys + off_y) * OUT_STRIDE * orig_h / IMG_H).squeeze(0).cpu().numpy()
    bw = (ws * OUT_STRIDE * orig_w / IMG_W).squeeze(0).cpu().numpy()
    bh = (hs * OUT_STRIDE * orig_h / IMG_H).squeeze(0).cpu().numpy()
    sc = scores.squeeze(0).cpu().numpy()

    detections = []
    for i in range(max_dets):
        if sc[i] < conf_thresh:
            break
        detections.append({
            'center_x' : float(cx[i]),
            'center_y' : float(cy[i]),
            'bbox_w'   : float(bw[i]),
            'bbox_h'   : float(bh[i]),
            'angle_deg': float(angles[i]),
            'score'    : float(sc[i]),
        })
    return detections


print('Decoder ready.')

## 11 · Training Loop
- Cosine LR with linear warmup
- Saved by best validation circular MAE
- Per-head loss breakdown printed every 10 epochs

In [ ]:
CKPT_PATH = '/content/centernet_csl_best.pth'

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

# Cosine annealing with linear warmup
def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / (EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

best_mae = float('inf')
history  = {'train_loss': [], 'val_mae': [],
             'hm': [], 'wh': [], 'off': [], 'angle': []}


def validate():
    model.eval()
    all_pred, all_gt = [], []
    with torch.no_grad():
        for batch in val_loader:
            imgs = batch['image'].to(DEVICE)
            preds = model(imgs)
            orig_h, orig_w = 480, 640   # original image size

            dets = decode_predictions(preds, orig_w, orig_h, conf_thresh=0.1)
            gt_rows = df_val[df_val['image'] == batch['img_name'][0]]

            if dets and len(gt_rows):
                # Match by nearest center
                for det in dets:
                    dists = np.sqrt(
                        (gt_rows['center_x'] - det['center_x'])**2 +
                        (gt_rows['center_y'] - det['center_y'])**2)
                    closest = dists.idxmin()
                    if dists[closest] < 30:
                        all_pred.append(det['angle_deg'])
                        all_gt.append(gt_rows.loc[closest, 'angle_deg'])

    if not all_pred:
        return float('nan')
    pred_t = torch.tensor(all_pred)
    gt_t   = torch.tensor(all_gt)
    return circular_mae(pred_t, gt_t)


print(f'Training for {EPOCHS} epochs on {DEVICE}...')

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    sub_losses = {'hm': 0, 'wh': 0, 'off': 0, 'angle': 0}

    for batch in train_loader:
        imgs = batch['image'].to(DEVICE)
        optimizer.zero_grad()
        preds = model(imgs)
        loss, sub = compute_loss(preds, batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_loss += loss.item()
        for k in sub_losses: sub_losses[k] += sub[k]

    scheduler.step()
    n = len(train_loader)
    avg_loss = total_loss / n
    history['train_loss'].append(avg_loss)
    for k in sub_losses: history[k].append(sub_losses[k] / n)

    val_mae = validate()
    history['val_mae'].append(val_mae)

    if not math.isnan(val_mae) and val_mae < best_mae:
        best_mae = val_mae
        torch.save(model.state_dict(), CKPT_PATH)

    if epoch % 10 == 0 or epoch == 1:
        lr_now = optimizer.param_groups[0]['lr']
        print(f'Ep {epoch:3d}/{EPOCHS} | '
              f'loss={avg_loss:.4f} '
              f'(hm={sub_losses["hm"]/n:.3f} '
              f'wh={sub_losses["wh"]/n:.3f} '
              f'off={sub_losses["off"]/n:.3f} '
              f'ang={sub_losses["angle"]/n:.3f}) | '
              f'val_MAE={val_mae:.2f}° | '
              f'best={best_mae:.2f}° | '
              f'lr={lr_now:.2e}')

print(f'\nTraining done. Best val MAE: {best_mae:.2f}°')

## 12 · Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history['train_loss'], color='steelblue')
axes[0].set_title('Total train loss')
axes[0].set_xlabel('Epoch')

for k, color in [('hm','blue'),('wh','green'),('off','purple'),('angle','darkorange')]:
    axes[1].plot(history[k], label=k, color=color, alpha=0.85)
axes[1].set_title('Per-head losses')
axes[1].set_xlabel('Epoch')
axes[1].legend()

val_maes = [m for m in history['val_mae'] if not math.isnan(m)]
axes[2].plot(val_maes, color='darkorange')
axes[2].axhline(best_mae, color='red', linestyle='--',
                label=f'Best: {best_mae:.1f}°')
axes[2].axhline(39.57, color='gray', linestyle=':',
                label='Baseline: 39.57°')
axes[2].set_title('Val circular MAE (degrees)')
axes[2].set_xlabel('Epoch')
axes[2].legend()

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=130)
plt.show()
print(f'Improvement over baseline: {39.57 - best_mae:.2f}°')

## 13 · Full Evaluation

In [ ]:
from scipy.optimize import linear_sum_assignment

# Load best checkpoint
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()

INFER_TF = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

def infer_image(img_path, conf=0.20):
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    orig_h, orig_w = img_rgb.shape[:2]
    resized = cv2.resize(img_rgb, (IMG_W, IMG_H))
    inp     = INFER_TF(resized).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        preds = model(inp)
    return decode_predictions(preds, orig_w, orig_h, conf_thresh=conf)


def circ_diff(a, b):
    d = abs(a - b) % 360
    return min(d, 360 - d)


def evaluate(img_list, df_gt, conf=0.20, center_thresh=20):
    tp = fp = fn = 0
    center_errs, angle_errs = [], []

    for img_name in img_list:
        gt  = df_gt[df_gt['image'] == img_name][
              ['center_x','center_y','angle_deg']].values
        det = infer_image(os.path.join(IMG_DIR, img_name), conf)

        if len(det) == 0 and len(gt) == 0: continue
        if len(det) == 0: fn += len(gt); continue
        if len(gt)  == 0: fp += len(det); continue

        pred_cx = np.array([d['center_x'] for d in det])
        pred_cy = np.array([d['center_y'] for d in det])
        cost = np.sqrt((pred_cx[:,None]-gt[:,0])**2 +
                       (pred_cy[:,None]-gt[:,1])**2)
        ri, ci = linear_sum_assignment(cost)

        mp, mg = set(), set()
        for r, c in zip(ri, ci):
            if cost[r, c] <= center_thresh:
                tp += 1; mp.add(r); mg.add(c)
                center_errs.append(cost[r, c])
                angle_errs.append(circ_diff(det[r]['angle_deg'], gt[c, 2]))
        fp += len(det) - len(mp)
        fn += len(gt)  - len(mg)

    prec = tp / (tp + fp + 1e-9)
    rec  = tp / (tp + fn + 1e-9)
    f1   = 2*prec*rec / (prec + rec + 1e-9)
    mae  = np.mean(angle_errs)  if angle_errs  else float('nan')
    cmae = np.mean(center_errs) if center_errs else float('nan')
    p15  = np.mean(np.array(angle_errs) < 15) * 100 if angle_errs else 0
    p30  = np.mean(np.array(angle_errs) < 30) * 100 if angle_errs else 0
    return dict(TP=tp, FP=fp, FN=fn, Precision=prec, Recall=rec, F1=f1,
                Mean_Center_px=cmae, Angle_MAE=mae,
                Pct_15=p15, Pct_30=p30,
                angle_errs=angle_errs, center_errs=center_errs)


print('Evaluating on validation set...')
m = evaluate(val_imgs, df)
print('\n========== RESULTS ==========')
for k, v in m.items():
    if k not in ('angle_errs','center_errs'):
        print(f'  {k:22s}: {v:.4f}' if isinstance(v,float) else f'  {k:22s}: {v}')
print('================================')
print(f'  Baseline MAE  : 39.57°')
print(f'  CenterNet+CSL : {m["Angle_MAE"]:.2f}°')
print(f'  Improvement   : {39.57 - m["Angle_MAE"]:.2f}°')

## 14 · Error Analysis

In [ ]:
ae = np.array(m['angle_errs'])
ce = np.array(m['center_errs'])

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Angle error histogram
axes[0,0].hist(ae, bins=36, color='steelblue', edgecolor='none')
axes[0,0].axvline(ae.mean(), color='red', linestyle='--', label=f'Mean={ae.mean():.1f}°')
axes[0,0].axvline(15, color='orange', linestyle=':', label='15° threshold')
axes[0,0].set_title('Angle error distribution'); axes[0,0].legend()

# CDF
sae = np.sort(ae)
cdf = np.arange(1, len(sae)+1) / len(sae)
axes[0,1].plot(sae, cdf, 'steelblue', lw=2)
for t, col in [(15,'red'),(30,'orange'),(45,'green')]:
    axes[0,1].axvline(t, color=col, linestyle='--', label=f'{t}°')
axes[0,1].set_title('CDF of angle error'); axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

# Center error
axes[0,2].hist(ce, bins=20, color='purple', edgecolor='none')
axes[0,2].set_title('Center localization error (px)')

# Threshold bar
thresholds = [5,10,15,20,30,45]
pcts = [np.mean(ae < t)*100 for t in thresholds]
bars = axes[1,0].bar([f'{t}°' for t in thresholds], pcts,
                      color='steelblue', edgecolor='none')
axes[1,0].set_title('% within angle threshold'); axes[1,0].set_ylim(0,110)
for bar, pct in zip(bars, pcts):
    axes[1,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                   f'{pct:.0f}%', ha='center', fontsize=9)

# CSL confidence inspection — histogram of top-bin softmax probability
# (high confidence = peaked distribution = correct answer expected)
model.eval()
softmax_peaks = []
with torch.no_grad():
    for img_name in val_imgs[:10]:
        det = infer_image(os.path.join(IMG_DIR, img_name), conf=0.1)
        # Re-run to get raw logits for inspection
        img_bgr = cv2.imread(os.path.join(IMG_DIR, img_name))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        resized = cv2.resize(img_rgb, (IMG_W, IMG_H))
        inp = INFER_TF(resized).unsqueeze(0).to(DEVICE)
        out = model(inp)
        hm_nms = heatmap_nms(out['hm'])
        peaks  = (hm_nms.squeeze() > 0.1).nonzero()
        for pk in peaks:
            ang_logit = out['angle'][0, :, pk[0], pk[1]]
            prob = torch.softmax(ang_logit, dim=0).max().item()
            softmax_peaks.append(prob)

if softmax_peaks:
    axes[1,1].hist(softmax_peaks, bins=20, color='darkorange', edgecolor='none')
    axes[1,1].set_title('CSL peak softmax confidence\n(closer to 1 = sharper = better)')
    axes[1,1].set_xlabel('Max softmax probability')

# Polar error by GT angle
ax_pol = fig.add_subplot(2, 3, 6, projection='polar')
gt_ang_matched = []
for img_name in val_imgs:
    gt = df[df['image']==img_name][['center_x','center_y','angle_deg']].values
    det = infer_image(os.path.join(IMG_DIR, img_name))
    if not det or len(gt)==0: continue
    pcx = np.array([d['center_x'] for d in det])
    pcy = np.array([d['center_y'] for d in det])
    cost = np.sqrt((pcx[:,None]-gt[:,0])**2+(pcy[:,None]-gt[:,1])**2)
    ri,ci = linear_sum_assignment(cost)
    for r,c in zip(ri,ci):
        if cost[r,c]<=20: gt_ang_matched.append(gt[c,2])

gt_ang_matched = np.array(gt_ang_matched[:len(ae)])
if len(gt_ang_matched):
    ax_pol.scatter(np.radians(gt_ang_matched), ae, alpha=0.4, s=15, c='steelblue')
    ax_pol.set_title('Error by GT angle direction', pad=14)

plt.suptitle(f'CenterNet+CSL Error Analysis | MAE={ae.mean():.2f}°  '
             f'(baseline 39.57°, improvement {39.57-ae.mean():.2f}°)', fontsize=12)
plt.tight_layout()
plt.savefig('/content/error_analysis.png', dpi=130)
plt.show()

## 15 · Qualitative Visualisation

In [ ]:
def draw_tube(img, cx, cy, angle_deg, color, length=40, label=None):
    cx, cy = int(cx), int(cy)
    cv2.circle(img, (cx,cy), 8, color, -1)
    cv2.circle(img, (cx,cy), 8, (255,255,255), 1)
    rad = math.radians(angle_deg)
    ex  = int(cx + length * math.cos(rad))
    ey  = int(cy - length * math.sin(rad))
    cv2.arrowedLine(img, (cx,cy), (ex,ey), color, 2, tipLength=0.3)
    if label:
        cv2.putText(img, label, (cx+10,cy-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)

def visualise(img_name):
    img_path = os.path.join(IMG_DIR, img_name)
    img = cv2.imread(img_path)
    vis = img.copy()

    # GT — blue
    for _, row in df[df['image']==img_name].iterrows():
        draw_tube(vis, row['center_x'], row['center_y'],
                  row['angle_deg'], (255,100,0), label=f"{row['angle_deg']:.0f}°")

    # Predictions — green
    for d in infer_image(img_path):
        draw_tube(vis, d['center_x'], d['center_y'],
                  d['angle_deg'], (0,220,80), label=f"{d['angle_deg']:.0f}°")

    cv2.putText(vis,'GT:blue  Pred:green',(6,18),
                cv2.FONT_HERSHEY_SIMPLEX,0.5,(255,255,255),1)
    return cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)


n = min(6, len(val_imgs))
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, name in zip(axes.flat, val_imgs[:n]):
    ax.imshow(visualise(name))
    ax.set_title(name[:30], fontsize=8)
    ax.axis('off')
plt.suptitle('CenterNet+CSL: Predictions (green) vs GT (blue)\n'
             'Arrow = joint→tab direction', fontsize=12)
plt.tight_layout()
plt.savefig('/content/qualitative.png', dpi=130)
plt.show()

## 16 · Save Predictions & Final Report

In [ ]:
all_preds = []
for img_name in df['image'].unique():
    for d in infer_image(os.path.join(IMG_DIR, img_name)):
        d['image'] = img_name
        all_preds.append(d)

pred_df = pd.DataFrame(all_preds)
pred_df.to_csv('/content/predictions_centernet_csl.csv', index=False)

print('='*56)
print('     CENTERNET + CSL FINAL REPORT')
print('='*56)
print(f'  Architecture  : CenterNet (custom enc-dec) + CSL')
print(f'  Angle bins    : {N_ANGLE_BINS}  (1°/bin, σ={CSL_SIGMA}°)')
print(f'  Input size    : {IMG_W}×{IMG_H}')
print(f'  Parameters    : {n_params:,}')
print()
print(f'  Detection (centre ≤20px):')
print(f'    Precision   : {m["Precision"]:.3f}')
print(f'    Recall      : {m["Recall"]:.3f}')
print(f'    F1          : {m["F1"]:.3f}')
print(f'    TP/FP/FN    : {m["TP"]}/{m["FP"]}/{m["FN"]}')
print()
print(f'  Orientation:')
print(f'    Mean MAE    : {m["Angle_MAE"]:.2f}°')
print(f'    Within 15°  : {m["Pct_15"]:.1f}%')
print(f'    Within 30°  : {m["Pct_30"]:.1f}%')
print()
print(f'  vs Two-stage baseline: 39.57°')
print(f'  Improvement         : {39.57-m["Angle_MAE"]:.2f}° reduction')
print('='*56)